In [17]:
from IPython.display import HTML, display

html_content = """
<div style="font-family:Segoe UI;padding:20px;border:2px solid #0d6efd;
            border-radius:12px;background:#f8f9fa;max-width:1000px;">

    <h2 style="color:#0d6efd;">🤖 Two-Agent AutoGen Workflow</h2>

    <p>
        This notebook demonstrates a <b>Conversable Multi-Agent System</b> using AutoGen,
        Gemini, AgentOps, and a Local Code Executor.
    </p>

    <hr>

    <h3>Actors Involved</h3>

    <div style="display:flex;gap:20px;">

        <div style="flex:1;padding:15px;background:white;border-left:5px solid #198754;">
            <h4>Agent 1</h4>
            <b>Code Executor Agent</b><br>
            Executes Python code generated by another agent.<br>
            Uses LocalCommandLineCodeExecutor.
        </div>

        <div style="flex:1;padding:15px;background:white;border-left:5px solid #fd7e14;">
            <h4>Agent 2</h4>
            <b>Code Writer Agent</b><br>
            Uses Gemini LLM to generate Python code and solve tasks.
        </div>

    </div>

    <br>

    <h3>Workflow</h3>

    <div style="background:white;padding:15px;border-radius:8px;">

        <pre style="font-size:15px;">
User
 │
 ▼
Code Executor Agent
 │
 ▼
Code Writer Agent (Gemini)
 │
 ▼
Generates Python Code
 │
 ▼
Code Executor Agent
 │
 ▼
Executes the Code
 │
 ▼
Returns Result
        </pre>

    </div>

    <br>

    <h3>Example Task</h3>

    <div style="background:white;padding:15px;border-left:5px solid #6f42c1;">

        User Request:
        <br><br>

        <i>
        "Calculate the number of unique permutations of the word ALGEBRA"
        </i>

        <br><br>

        Gemini generates optimized Python code:

        <br><br>

        Unique Permutations = 7! / 2!

        <br><br>

        Result:
        <b>2520</b>

    </div>

    <br>

    <h3>Components Used</h3>

    <ul>
        <li>AutoGen Conversable Agents</li>
        <li>Gemini 3.6 Flash</li>
        <li>LocalCommandLineCodeExecutor</li>
        <li>AgentOps Monitoring</li>
        <li>Python Runtime</li>
    </ul>

    <br>

    <div style="background:#d1e7dd;
                color:#0f5132;
                padding:15px;
                border-radius:8px;
                font-weight:bold;">

        Flow Summary:
        User ➜ Code Executor Agent ➜ Gemini Code Writer Agent ➜ Python Code ➜ Execution ➜ Final Answer

    </div>

</div>
"""

display(HTML(html_content))

In [1]:
import autogen
autogen.__version__

'0.14.1'

In [2]:
keys = {}

with open("secrets.txt", "r") as f:
    for line in f:
        key, value = line.strip().split("=", 1)
        keys[key] = value

agentops_api_key = keys["AGENTOPS_API_KEY"]
openai_api_key = keys["OPENAI_API_KEY"]
gemini_api_key = keys["GEMINI_API_KEY"]

In [3]:
import agentops

session = agentops.init(agentops_api_key)
print(session)

🖇 AgentOps: [ConcurrentFuturesInstrumentor] Successfully instrumented concurrent.futures.ThreadPoolExecutor


None


🖇 AgentOps: You're on the agentops free plan 🤔
--- Logging error ---
Traceback (most recent call last):
  File "C:\Users\xsanthkum\AppData\Local\anaconda3\Lib\logging\__init__.py", line 1154, in emit
    stream.write(msg + self.terminator)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\xsanthkum\AppData\Local\anaconda3\Lib\encodings\cp1252.py", line 19, in encode
    return codecs.charmap_encode(input,self.errors,encoding_table)[0]
           ~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
UnicodeEncodeError: 'charmap' codec can't encode character '\U0001f914' in position 66: character maps to <undefined>
Call stack:
  File "C:\Users\xsanthkum\AppData\Local\anaconda3\Lib\runpy.py", line 198, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "C:\Users\xsanthkum\AppData\Local\anaconda3\Lib\runpy.py", line 88, in _run_code
    exec(code, run_globals)
  File "C:\Users\xsanthkum\AppData\Local\anaconda3\Lib\site-packages\ipykernel_launcher.py", li

In [4]:
from pathlib import Path

from autogen import ConversableAgent
from autogen.coding import  CodeBlock, LocalCommandLineCodeExecutor
work_dir = Path("coding")
work_dir.mkdir(exist_ok=True)

executor = LocalCommandLineCodeExecutor(work_dir=work_dir)

#Agent 1
code_executor_agent = ConversableAgent(
    name="code_executor_agent",
    llm_config=False,
    code_execution_config={
        "executor": executor,
    },
    human_input_mode="NEVER",
)

In [5]:
# The code writer agent's system message is to instruct the LLM on how to
# use the Jupyter code executor with IPython kernel.
code_writer_system_message = """
You have been given coding capability to solve tasks using Python code.
In the following cases, suggest python code (in a python coding block) or shell script (in a sh coding block) for the user to execute.
    1. When you need to collect info, use the code to output the info you need, for example, browse or search the web, download/read a file, 
    print the content of a webpage or a file, get the current date/time, check the operating system. After sufficient info is printed and the 
    task is ready to be solved based on your language skill, you can solve the task by yourself.
    2. When you need to perform some task with code, use the code to perform the task and output the result. Finish the task smartly.
Solve the task step by step if you need to. If a plan is not provided, explain your plan first. Be clear which step uses code, and which step uses your 
language skill.When using code, you must indicate the script type in the code block. The user cannot provide any other feedback or perform any other 
action beyond executing the code you suggest. The user can't modify your code. So do not suggest incomplete code which requires users to modify. 
Don't use a code block if it's not intended to be executed by the user.
If you want the user to save the code in a file before executing it, put # filename: <filename> inside the code block as the first line. Don't include multiple code blocks in one response. Do not ask users to copy and paste the result. Instead, use 'print' function for the output when relevant. Check the execution result returned by the user.
"""

import os

#Agent 2

code_writer_agent = ConversableAgent(
    "code_writer",
    system_message=code_writer_system_message,
    llm_config={"config_list": [{
    "model": "models/gemini-flash-lite-latest",
    "api_key": gemini_api_key,
    "api_type": "google"
}]},
    code_execution_config=False,  # Turn off code execution for this agent.
    max_consecutive_auto_reply=2,
    human_input_mode="NEVER",)

In [6]:
import pprint

chat_result = code_executor_agent.initiate_chat(
    code_writer_agent, message=" write python code for permutation for the word ALGEbRA. use optimised way to calculate it .I want final result count."
)

pprint.pprint(chat_result)

code_executor_agent (to code_writer):

 write python code for permutation for the word ALGEbRA. use optimised way to calculate it .I want final result count.

--------------------------------------------------------------------------------
code_writer (to code_executor_agent):

To find the number of unique permutations of the word "ALGEbRA" efficiently, we can use the formula for permutations of a multiset. 

The word has 7 letters in total, with some letters repeating. The formula is:
$$\text{Number of permutations} = \frac{n!}{n_1! \times n_2! \times \dots \times n_k!}$$
where $n$ is the total number of items, and $n_1, n_2, \dots, n_k$ are the frequencies of each distinct item.

Let's write a Python script using `math.factorial` and `collections.Counter` to calculate this accurately and quickly.

```python
# filename: permutations_count.py
from collections import Counter
import math

word = "ALGEbRA"
# Convert to lowercase or uppercase to treat case-insensitively if needed, 
# but t

C:\Users\xsanthkum\AppData\Local\anaconda3\Lib\site-packages\autogen\oai\gemini.py:1131: UserWarning: Cost calculation is not implemented for model models/gemini-flash-lite-latest. Cost will be calculated zero.
  warnings.warn(


code_executor_agent (to code_writer):

exitcode: 0 (execution succeeded)
Code output: --- Case-sensitive calculation (exact characters) ---
'A': 2
'L': 1
'G': 1
'E': 1
'b': 1
'R': 1
Total unique permutations (case-sensitive): 2520

--- Case-insensitive calculation (treating 'A' and 'a' as the same) ---
'a': 2
'l': 1
'g': 1
'e': 1
'b': 1
'r': 1
Total unique permutations (case-insensitive): 2520


--------------------------------------------------------------------------------
code_writer (to code_executor_agent):

Based on the optimized calculation using multiset permutations ($\frac{n!}{n_1! n_2! \dots}$), the word **ALGEbRA** contains 7 letters in total, with the letter 'A'/'a' repeating 2 times and all other letters appearing once.

The final result count for the unique permutations is:
* **2,520**

--------------------------------------------------------------------------------
code_executor_agent (to code_writer):



-----------------------------------------------------------------

🖇 AgentOps: [agentops.InternalSpanProcessor] Error uploading logfile: Failed to upload logfile: HTTPSConnectionPool(host='api.agentops.ai', port=443): Max retries exceeded with url: /v4/logs/upload/ (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1028)')))


ChatResult(chat_id=99126182697416458402468255325363919879,
           chat_history=[{'content': ' write python code for permutation for '
                                     'the word ALGEbRA. use optimised way to '
                                     'calculate it .I want final result count.',
                          'name': 'code_executor_agent',
                          'role': 'assistant'},
                         {'content': 'To find the number of unique '
                                     'permutations of the word "ALGEbRA" '
                                     'efficiently, we can use the formula for '
                                     'permutations of a multiset. \n'
                                     '\n'
                                     'The word has 7 letters in total, with '
                                     'some letters repeating. The formula is:\n'
                                     '$$\\text{Number of permutations} = '
                           

In [7]:
import pprint

chat_result = code_executor_agent.initiate_chat(
    code_writer_agent, message="""Write optimized Python code to calculate the number of unique permutations of the word 'ALGEBRA.'
    For example, the word 'ALGEbRA' should have 2520 unique permutations. Use efficient logic to compute the result and verify the output using 
    the word 'ALGEBRA.' The code should only return the final permutation count.""")
pprint.pprint(chat_result)
agentops.end_session("Success")

code_executor_agent (to code_writer):

Write optimized Python code to calculate the number of unique permutations of the word 'ALGEBRA.'
    For example, the word 'ALGEbRA' should have 2520 unique permutations. Use efficient logic to compute the result and verify the output using 
    the word 'ALGEBRA.' The code should only return the final permutation count.

--------------------------------------------------------------------------------
code_writer (to code_executor_agent):

I will write a Python script to calculate the number of unique permutations of the word 'ALGEBRA' using the formula for permutations of a multiset ($n! / (n_1! \cdot n_2! \cdot ... \cdot n_k!)$), and print the final result.

```python
# filename: calculate_permutations.py
from collections import Counter
import math

def count_unique_permutations(word):
    # Total number of letters
    total_letters = len(word)
    
    # Count frequencies of each letter
    counts = Counter(word)
    
    # Calculate denominat

🖇 AgentOps: end_session() is deprecated and will be removed in v4 in the future. Use agentops.end_trace() instead.
🖇 AgentOps: end_session called but no active trace context found.


ChatResult(chat_id=273035725058160530317008441797227090967,
           chat_history=[{'content': 'Write optimized Python code to calculate '
                                     'the number of unique permutations of the '
                                     "word 'ALGEBRA.'\n"
                                     "    For example, the word 'ALGEbRA' "
                                     'should have 2520 unique permutations. '
                                     'Use efficient logic to compute the '
                                     'result and verify the output using \n'
                                     "    the word 'ALGEBRA.' The code should "
                                     'only return the final permutation count.',
                          'name': 'code_executor_agent',
                          'role': 'assistant'},
                         {'content': 'I will write a Python script to '
                                     'calculate the number of unique '
      

In [8]:
dir(chat_result)


['__annotations__',
 '__class__',
 '__dataclass_fields__',
 '__dataclass_params__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__firstlineno__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__match_args__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__replace__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__static_attributes__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 'chat_history',
 'chat_id',
 'cost',
 'human_input',
 'summary']

In [9]:
chat_result.chat_history

[{'content': "Write optimized Python code to calculate the number of unique permutations of the word 'ALGEBRA.'\n    For example, the word 'ALGEbRA' should have 2520 unique permutations. Use efficient logic to compute the result and verify the output using \n    the word 'ALGEBRA.' The code should only return the final permutation count.",
  'role': 'assistant',
  'name': 'code_executor_agent'},
 {'content': "I will write a Python script to calculate the number of unique permutations of the word 'ALGEBRA' using the formula for permutations of a multiset ($n! / (n_1! \\cdot n_2! \\cdot ... \\cdot n_k!)$), and print the final result.\n\n```python\n# filename: calculate_permutations.py\nfrom collections import Counter\nimport math\n\ndef count_unique_permutations(word):\n    # Total number of letters\n    total_letters = len(word)\n    \n    # Count frequencies of each letter\n    counts = Counter(word)\n    \n    # Calculate denominator: product of factorials of each letter's frequency\n

In [10]:
chat_result.chat_id

273035725058160530317008441797227090967

In [11]:
chat_result.cost

{'usage_including_cached_inference': {'total_cost': 0.0,
  'models/gemini-flash-lite-latest': {'cost': 0.0,
   'prompt_tokens': 2725,
   'completion_tokens': 979,
   'total_tokens': 3704}},
 'usage_excluding_cached_inference': {'total_cost': 0.0,
  'models/gemini-flash-lite-latest': {'cost': 0.0,
   'prompt_tokens': 2725,
   'completion_tokens': 979,
   'total_tokens': 3704}}}

In [12]:
chat_result.human_input

[]

In [13]:
chat_result.summary

''